# Bayan Student Notebook — 01_text_processing_tokenization.ipynb

**Student:** Anas Ibrahim Al-Mutairi  
**GitHub:** `Ans212121`  
**Source:** reorganised from my own executed combined Bayan Colab notebook.

Run top-to-bottom in Google Colab and preserve the saved outputs before final submission.


In [5]:
# تجهيز بيئة التشغيل / Setup environment

import importlib.util
import subprocess
import sys

required_versions = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "spacy": "3.8.7"
}

packages_to_install = []

for package_name, package_version in required_versions.items():
    if importlib.util.find_spec(package_name) is None:
        packages_to_install.append(
            f"{package_name}=={package_version}"
        )

if packages_to_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *packages_to_install
        ]
    )

print("Python version:", sys.version.split()[0])
print("Environment setup completed")

Python version: 3.13.15
Environment setup completed


In [6]:
# Core imports for text preprocessing and tokenisation

import html
import re
import unicodedata
from dataclasses import dataclass
from typing import Iterable

import numpy as np

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing


# Keep random operations reproducible
RANDOM_SEED = 42
random_generator = np.random.default_rng(RANDOM_SEED)


# Small synthetic dataset used for testing
test_texts = [
    "أهلاً وسهلاً بكم في برنامج بيان!",
    "مرحبــاً\u00a0بكم",
    "Contact us at learner@example.org",
    "للتجربة فقط: 0551234567",
    "Natural language processing connects text and models."
]

print("Number of test samples:", len(test_texts))

Number of test samples: 5


In [7]:
# Inspect Unicode information for each character

def get_unicode_details(text: str) -> list[dict[str, str]]:
    details = []

    for character in text:
        details.append({
            "character": character,
            "unicode_code": f"U+{ord(character):04X}",
            "unicode_name": unicodedata.name(character, "UNKNOWN")
        })

    return details


test_characters = "أé"
unicode_info = get_unicode_details(test_characters)

for item in unicode_info:
    print(item)

assert unicode_info[0]["unicode_code"] == "U+0623"

print("Unicode inspection=PASS")

{'character': 'أ', 'unicode_code': 'U+0623', 'unicode_name': 'ARABIC LETTER ALEF WITH HAMZA ABOVE'}
{'character': 'é', 'unicode_code': 'U+00E9', 'unicode_name': 'LATIN SMALL LETTER E WITH ACUTE'}
Unicode inspection=PASS


In [8]:
# Text preparation and privacy protection

DIACRITIC_PATTERN = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)

TATWEEL_CHAR = "\u0640"
SPACE_PATTERN = re.compile(r"\s+")
HTML_PATTERN = re.compile(r"<[^>]+>")

EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)"
)


@dataclass(frozen=True)
class TextRecord:
    raw_text: str
    model_text: str


def hide_private_info(text: str) -> str:
    cleaned_text = EMAIL_PATTERN.sub("<EMAIL>", text)
    cleaned_text = PHONE_PATTERN.sub("<PHONE>", cleaned_text)
    return cleaned_text


def prepare_text(
    text: str,
    remove_diacritics: bool = False,
    normalize_alef: bool = False
) -> TextRecord:

    if not isinstance(text, str):
        raise TypeError("Input must be a string")

    original_text = text

    prepared_text = html.unescape(text)
    prepared_text = unicodedata.normalize("NFC", prepared_text)
    prepared_text = HTML_PATTERN.sub(" ", prepared_text)
    prepared_text = prepared_text.replace(TATWEEL_CHAR, "")

    if remove_diacritics:
        prepared_text = DIACRITIC_PATTERN.sub("", prepared_text)

    if normalize_alef:
        prepared_text = re.sub(r"[إأآٱ]", "ا", prepared_text)

    prepared_text = hide_private_info(prepared_text)
    prepared_text = SPACE_PATTERN.sub(" ", prepared_text).strip()

    return TextRecord(
        raw_text=original_text,
        model_text=prepared_text
    )


records = [prepare_text(text) for text in test_texts]

for record in records:
    print({
        "raw": record.raw_text,
        "model": record.model_text
    })


assert records[1].raw_text != records[1].model_text
assert "learner@example.org" not in records[2].model_text
assert "0551234567" not in records[3].model_text
assert records[2].raw_text == test_texts[2]

print("Two-copy preprocessing contract=PASS")

{'raw': 'أهلاً وسهلاً بكم في برنامج بيان!', 'model': 'أهلاً وسهلاً بكم في برنامج بيان!'}
{'raw': 'مرحبــاً\xa0بكم', 'model': 'مرحباً بكم'}
{'raw': 'Contact us at learner@example.org', 'model': 'Contact us at <EMAIL>'}
{'raw': 'للتجربة فقط: 0551234567', 'model': 'للتجربة فقط: <PHONE>'}
{'raw': 'Natural language processing connects text and models.', 'model': 'Natural language processing connects text and models.'}
Two-copy preprocessing contract=PASS


In [9]:
import spacy

# Create a lightweight multilingual spaCy pipeline
sentence_processor = spacy.blank("xx")
sentence_processor.add_pipe("sentencizer")


def split_sentences(text: str) -> list[str]:
    cleaned_text = prepare_text(text).model_text

    return [
        sentence.text.strip()
        for sentence in sentence_processor(cleaned_text).sents
        if sentence.text.strip()
    ]


examples = {
    "ar": "الخدمة جيدة. لم يصل الرمز!",
    "en": "The portal stopped. Please retry."
}

sentence_results = {
    language: split_sentences(text)
    for language, text in examples.items()
}

for language, sentences in sentence_results.items():
    print(language, sentences)


abbreviation_test = split_sentences(
    "راجع د. أحمد. ثم أعد المحاولة."
)

print(
    "Abbreviation test:",
    abbreviation_test
)


assert sentence_results["ar"] == [
    "الخدمة جيدة.",
    "لم يصل الرمز!"
]

assert sentence_results["en"] == [
    "The portal stopped.",
    "Please retry."
]

print("SPACY_SENTENCE_PIPELINE=PASS")

ar ['الخدمة جيدة.', 'لم يصل الرمز!']
en ['The portal stopped.', 'Please retry.']
Abbreviation test: ['راجع د.', 'أحمد.', 'ثم أعد المحاولة.']
SPACY_SENTENCE_PIPELINE=PASS


In [10]:
# Build a simple local WordPiece tokenizer

special_tokens = [
    "[PAD]",
    "[UNK]",
    "[CLS]",
    "[SEP]",
    "[MASK]"
]

base_tokens = [
    "أهلاً",
    "وسهلاً",
    "بكم",
    "في",
    "برنامج",
    "بيان",
    "!",
    "مرحبا",
    "Contact",
    "us",
    "at",
    "<",
    "EMAIL",
    ">",
    "Natural",
    "language",
    "processing",
    "connects",
    "text",
    "and",
    "models",
    "."
]

all_tokens = special_tokens + base_tokens

token_to_id = {
    token: index
    for index, token in enumerate(all_tokens)
}

wordpiece_tokenizer = Tokenizer(
    WordPiece(
        vocab=token_to_id,
        unk_token="[UNK]"
    )
)

wordpiece_tokenizer.pre_tokenizer = BertPreTokenizer()

wordpiece_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[
        ("[CLS]", token_to_id["[CLS]"]),
        ("[SEP]", token_to_id["[SEP]"])
    ]
)

for record in records:
    tokenized = wordpiece_tokenizer.encode(record.model_text)

    print("Text:", record.model_text)
    print("Tokens:", tokenized.tokens)
    print("IDs:", tokenized.ids)
    print("-" * 30)

test_encoding = wordpiece_tokenizer.encode("مرحبا بكم")

assert test_encoding.tokens == [
    "[CLS]",
    "مرحبا",
    "بكم",
    "[SEP]"
]

print("Local WordPiece demonstration=PASS")

Text: أهلاً وسهلاً بكم في برنامج بيان!
Tokens: ['[CLS]', 'أهلاً', 'وسهلاً', 'بكم', 'في', 'برنامج', 'بيان', '!', '[SEP]']
IDs: [2, 5, 6, 7, 8, 9, 10, 11, 3]
------------------------------
Text: مرحباً بكم
Tokens: ['[CLS]', '[UNK]', 'بكم', '[SEP]']
IDs: [2, 1, 7, 3]
------------------------------
Text: Contact us at <EMAIL>
Tokens: ['[CLS]', 'Contact', 'us', 'at', '<', 'EMAIL', '>', '[SEP]']
IDs: [2, 13, 14, 15, 16, 17, 18, 3]
------------------------------
Text: للتجربة فقط: <PHONE>
Tokens: ['[CLS]', '[UNK]', '[UNK]', '[UNK]', '<', '[UNK]', '>', '[SEP]']
IDs: [2, 1, 1, 1, 16, 1, 18, 3]
------------------------------
Text: Natural language processing connects text and models.
Tokens: ['[CLS]', 'Natural', 'language', 'processing', 'connects', 'text', 'and', 'models', '.', '[SEP]']
IDs: [2, 19, 20, 21, 22, 23, 24, 25, 26, 3]
------------------------------
Local WordPiece demonstration=PASS


In [11]:
# Measure token fertility and truncation rate

def count_words(text: str) -> int:
    return max(1, len(text.split()))


def calculate_token_fertility(
    tokenizer: Tokenizer,
    text: str
) -> float:

    encoded_tokens = tokenizer.encode(text).tokens

    content_tokens = [
        token
        for token in encoded_tokens
        if token not in {"[CLS]", "[SEP]", "[PAD]"}
    ]

    return len(content_tokens) / count_words(text)


def calculate_truncation_rate(
    tokenizer: Tokenizer,
    texts: Iterable[str],
    max_length: int
) -> float:

    text_list = list(texts)

    if not text_list:
        raise ValueError("Text list must not be empty")

    too_long = sum(
        len(tokenizer.encode(text).ids) > max_length
        for text in text_list
    )

    return too_long / len(text_list)


model_texts = [
    record.model_text
    for record in records
]

fertilities = [
    calculate_token_fertility(wordpiece_tokenizer, text)
    for text in model_texts
]

rate_at_10 = calculate_truncation_rate(
    wordpiece_tokenizer,
    model_texts,
    max_length=10
)

print(
    "Token fertility per sample:",
    [round(value, 2) for value in fertilities]
)

print(
    "Average token fertility:",
    round(float(np.mean(fertilities)), 2)
)

print(
    "Truncation rate at length 10:",
    f"{rate_at_10:.0%}"
)

assert all(value > 0 for value in fertilities)
assert 0.0 <= rate_at_10 <= 1.0

print("Tokenisation metrics=PASS")

Token fertility per sample: [1.17, 1.0, 1.5, 2.0, 1.14]
Average token fertility: 1.36
Truncation rate at length 10: 0%
Tokenisation metrics=PASS


In [12]:
# Padding, truncation, and simple embeddings

MAX_SEQUENCE_LENGTH = 12

wordpiece_tokenizer.enable_truncation(
    max_length=MAX_SEQUENCE_LENGTH
)

wordpiece_tokenizer.enable_padding(
    length=MAX_SEQUENCE_LENGTH,
    pad_id=token_to_id["[PAD]"],
    pad_token="[PAD]"
)

encoded_batch = [
    wordpiece_tokenizer.encode(text)
    for text in model_texts[:2]
]

input_ids = np.array(
    [item.ids for item in encoded_batch],
    dtype=np.int64
)

attention_mask = np.array(
    [item.attention_mask for item in encoded_batch],
    dtype=np.int64
)

embedding_matrix = random_generator.normal(
    0.0,
    0.02,
    size=(len(token_to_id), 8)
)

embeddings = embedding_matrix[input_ids]

print("Input IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Embeddings shape:", embeddings.shape)
print("First attention mask:", attention_mask[0].tolist())

assert input_ids.shape == (2, MAX_SEQUENCE_LENGTH)
assert attention_mask.shape == input_ids.shape
assert embeddings.shape == (2, MAX_SEQUENCE_LENGTH, 8)

print("IDs-to-embeddings pipeline=PASS")

Input IDs shape: (2, 12)
Attention mask shape: (2, 12)
Embeddings shape: (2, 12, 8)
First attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
IDs-to-embeddings pipeline=PASS


In [13]:
# Optional comparison with multilingual BERT tokenizer

try:
    from transformers import AutoTokenizer

    multilingual_tokenizer = AutoTokenizer.from_pretrained(
        "google-bert/bert-base-multilingual-cased",
        use_fast=True
    )

    sample_text = (
        "معالجة اللغة الطبيعية مفيدة "
        "Natural language processing is useful"
    )

    bert_tokens = multilingual_tokenizer.tokenize(sample_text)

    print("mBERT tokens:", bert_tokens)
    print("mBERT token count:", len(bert_tokens))
    print("Fast tokenizer:", multilingual_tokenizer.is_fast)

except Exception as error:
    print("mBERT comparison skipped; Core pipeline can continue.")
    print("Reason:", type(error).__name__)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

mBERT tokens: ['مع', '##الجة', 'اللغة', 'الطبيعية', 'م', '##فيد', '##ة', 'Natural', 'language', 'processing', 'is', 'useful']
mBERT token count: 12
Fast tokenizer: True


In [14]:
# Final core checks

core_checks = {
    "unicode":
        unicode_info[0]["unicode_code"] == "U+0623",

    "spacy_sentence_pipeline":
        all(len(value) == 2 for value in sentence_results.values()),

    "raw_copy_preserved":
        records[2].raw_text == test_texts[2],

    "pii_masked":
        "<EMAIL>" in records[2].model_text
        and "<PHONE>" in records[3].model_text,

    "token_metrics":
        all(value > 0 for value in fertilities)
        and 0.0 <= rate_at_10 <= 1.0,

    "embedding_shape":
        embeddings.shape == (2, MAX_SEQUENCE_LENGTH, 8)
}

for check_name, passed in core_checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{check_name}: {status}")

assert all(core_checks.values())

print("DAY1_NOTEBOOK1_CORE=PASS")

unicode: PASS
spacy_sentence_pipeline: PASS
raw_copy_preserved: PASS
pii_masked: PASS
token_metrics: PASS
embedding_shape: PASS
DAY1_NOTEBOOK1_CORE=PASS
